# **Review and correction of pre-labeling dataset**

The main goal of this notebook is the review of *pre-labeling* process and the correction of the labels from **CVAT** of our dataset.

To achieve this goal we will follow the next steps:
* Load our *dataset* to **FiftyOne** to visualice the labels assigned in the *pre-labeling* process.
* Load our *samples* to **CVAT** to do the respectives corrections.
* Load our corrected labels to **FiftyOne**.
* Select the *samples* for our *validation* and *test* set.
* Export dataset final version for train.

## **Init Parameters**
```
DS_DIR = BASE_PROJECT_DIR/data/prc/ds_arg_ID_card_elements_v1
DS_EXPORT_DIR = BASE_PROJECT_DIR/data/ds/ds_arg_ID_card_elements_v1
DS_NAME = ds_arg_ID_card_elements_v1
PROJECT_NAME = "argentinian-id-card-elements-detection"
ANNO_KEY = "CVAT_RUN_00"
CLASSES = [
    "picture",
    "shield",
    "doc_number",
    "tramite_number",
    "pdf_417",
    "mrz",
    "address",
    "gender",
    "country",
]
```

## **Expected Input**
* *Pre-labeling dataset* was made in the previous step.

## **Expected Output**
* Dataset revised and corrected.
* Dataset splitted in *train*, *val* and *test* sets, ready to train a **YOLO** model for the task of ID-Card elements detection.

## **Example**
<table>
  <tr>
    <td align="center" style="padding: 10px;">
      <img src="./assets/00_5.png" width="420"><br>
      <em>Figure 1. Pre-Labeling image.</em>
    </td>
    <td align="center" style="padding: 10px;">
      <img src="./assets/00_4.png" width="420"><br>
      <em>Figura 2. Corrected ID-Card elements.</em>
    </td>
  </tr>
</table>

## **Libraries and Settings**

In [2]:
import os
import getpass
if not os.environ.get('FIFTYONE_CVAT_USERNAME'):
    os.environ['FIFTYONE_CVAT_USERNAME'] = input("CVAT Username: ")
if not os.environ.get('FIFTYONE_CVAT_PASSWORD'):
    os.environ['FIFTYONE_CVAT_PASSWORD'] = getpass.getpass("CVAT Password: ")
if not os.environ.get('FIFTYONE_CVAT_URL'):
    os.environ['FIFTYONE_CVAT_URL'] = "http://localhost:8080"
if not os.environ.get("BASE_PROJECT_DIR"):
    os.environ["BASE_PROJECT_DIR"] = input("Enter the base project directory path: ")
from pathlib import Path
import fiftyone as fo

DS_DIR = Path(os.environ.get("BASE_PROJECT_DIR")) / "data/prc/ds_arg_ID_card_elements_v1"
DS_EXPORT_DIR = Path(os.environ.get("BASE_PROJECT_DIR")) / "data/ds/ds_arg_ID_card_elements_v1"
DS_NAME = "ds_arg_ID_card_elements_v1"
CLASSES = ["picture", "shield", "doc_number", "tramite_number", "pdf_417", "mrz", "address", "gender", "country"]
ANNO_KEY = "CVAT_RUN_00"
PROJECT_NAME = "argentinian-id-card-elements-detection"

print(f"FiftyOne version: {fo.__version__}")

FiftyOne version: 1.21.0


* Load *dataset* to **FiftyOne**

In [16]:
if fo.dataset_exists(DS_NAME):
    print(f"Deleted existing dataset '{DS_NAME}'...")
    fo.delete_dataset(DS_NAME)
else: 
    dataset = fo.Dataset(DS_NAME, persistent=True)
    dataset.add_dir(
        dataset_dir=str(DS_DIR),
        dataset_type=fo.types.YOLOv5Dataset,
        split="train",
        classes=CLASSES
    )

dataset

Ignoring unsupported parameter 'classes' for importer type <class 'fiftyone.utils.yolo.YOLOv5DatasetImporter'>
 100% |█████████████████| 322/322 [528.5ms elapsed, 0s remaining, 610.9 samples/s]      


Name:        ds_arg_ID_card_elements_v1
Media type:  image
Num samples: 322
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)

* Load our *samples* to **CVAT**.

In [17]:
view = dataset

view.annotate(
    ANNO_KEY,
    project_name=PROJECT_NAME,
    label_field="ground_truth",
    label_type="detections",
    classes=CLASSES,
    image_quality=100,
    overwrite=True

)

view.get_annotation_info(ANNO_KEY)

Computing metadata...
 100% |█████████████████| 322/322 [38.5ms elapsed, 0s remaining, 8.4K samples/s] 
Uploading samples to CVAT...


{
    "key": "CVAT_RUN_00",
    "version": "1.20.1",
    "timestamp": "2026-08-24T11:54:39.513379",
    "config": {
        "cls": "fiftyone.utils.cvat.CVATBackendConfig",
        "type": "annotation",
        "method": "cvat",
        "overwrite": true,
        "name": "cvat",
        "label_schema": {
            "ground_truth": {
                "type": "detections",
                "classes": [
                    "picture",
                    "shield",
                    "doc_number",
                    "tramite_number",
                    "pdf_417",
                    "mrz",
                    "address",
                    "gender",
                    "country"
                ],
                "attributes": {},
                "existing_field": true,
                "allow_additions": true,
                "allow_deletions": true,
                "allow_label_edits": true,
                "allow_spatial_edits": true
            }
        },
        "media_field": "filep

* Load our corrected *samples* to our **FiftyOne** dataset.

In [4]:
dataset.load_annotations(ANNO_KEY)

Download complete
Loading labels for field 'ground_truth'...
 100% |█████████████████| 322/322 [440.3ms elapsed, 0s remaining, 731.4 samples/s]      


* Export our dataset for *train*.

In [21]:
splits = ["train", "val", "test"]

for split in splits:
    split_view = dataset.match_tags(split)
    print(f"{split}: {len(split_view)} samples")
    split_view.export(
        export_dir=str(DS_EXPORT_DIR),
        dataset_type=fo.types.YOLOv5Dataset,
        split=split,
        label_field="ground_truth",
        classes=CLASSES,
    )

train: 258 samples
 100% |█████████████████| 258/258 [991.4ms elapsed, 0s remaining, 253.4 samples/s]      
val: 32 samples
Directory '/home/nahuel/Documentos/Python/DNI/data/ds/ds_arg_ID_card_elements_v1' already exists; export will be merged with existing files
 100% |███████████████████| 32/32 [227.6ms elapsed, 0s remaining, 140.6 samples/s]     
test: 32 samples
Directory '/home/nahuel/Documentos/Python/DNI/data/ds/ds_arg_ID_card_elements_v1' already exists; export will be merged with existing files
 100% |███████████████████| 32/32 [120.5ms elapsed, 0s remaining, 265.6 samples/s]     
